# VoxCPM2 ONNX — Benchmark RTF trên Kaggle T4

Đo RTF của VoxCPM2 (2B) sau khi export sang ONNX, chạy cả GPU và CPU để so sánh.

## Pipeline

1. Cài `onnxruntime-gpu` bản CUDA 12 (Kaggle dùng CUDA 12, không phải 13)
2. Clone repo inference, tải ONNX models vào đúng thư mục repo
3. Tải VoxCPM2 base model vào cache (cần cho tokenizer, không nạp lên GPU)
4. Preprocessing bằng PyTorch trên CPU, lưu input ra `prepped.npz`
5. Benchmark ONNX, nạp session một lần rồi lặp qua các câu

## Bốn cái bẫy đã gặp và cách tránh

**Thứ tự nạp model.** `infer.py` gốc nạp ONNX sessions (17GB RAM) trước rồi mới nạp model PyTorch (8GB), hai thứ cùng tồn tại nên bị kill. Notebook này preprocessing xong, giải phóng model PyTorch, rồi mới nạp ONNX.

**Thứ tự `.cpu()` và `.to(float32)`.** `infer.py` gọi `model.to(torch.float32).cpu()`, tức convert bf16 sang fp32 ngay trên GPU làm 5.5GB phình thành 11GB rồi OOM. Phải `.cpu()` trước.

**onnxruntime-gpu sai bản CUDA.** Bản mặc định hiện tại dựng cho CUDA 13, thiếu `libcublasLt.so.13` nên âm thầm fallback về CPU mà không báo lỗi. Phải pin 1.22.0 và set `LD_LIBRARY_PATH` trỏ tới thư viện NVIDIA mà torch cài qua pip. Luôn nhìn dòng `decode_step:` trong output để biết có thật sự chạy GPU không.

**VRAM T4 không đủ cho cả hai model lớn.** `prefill` 8.36GB cộng `decode_step` 8.66GB là 17GB, vượt 14.56GB. Chỉ đẩy `decode_step` (chạy ~30 lần mỗi câu) và `vae` lên GPU, để `prefill` (chạy 1 lần) ở CPU.

## Cần chuẩn bị

Một file ref audio 3 đến 10 giây upload lên Kaggle Dataset, và transcript của nó. Sửa ở Cell 6.

In [ ]:
# ============================================================
# Cell 1: CÀI ĐẶT
# ============================================================
# Không cài torch thủ công. Cài đè sẽ làm hỏng bản dựng CUDA của Kaggle.
# onnxruntime-gpu phải là bản CUDA 12; bản mới nhất dựng cho CUDA 13 và sẽ
# fallback về CPU mà không báo lỗi.
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu==1.22.0
!pip install -q voxcpm librosa soundfile huggingface_hub "transformers>=4.50,<4.58"

import torch
print("torch      :", torch.__version__)
print("CUDA sẵn có:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print()
    print("Không thấy GPU. Nếu pip đã cài đè torch, chạy lệnh sau rồi restart kernel:")
    print("  !pip uninstall -y torch torchaudio torchvision")
    print("  !pip install torch torchaudio torchvision")

In [ ]:
# ============================================================
# Cell 2: HELPER + KIỂM TRA CUDA PROVIDER
# ============================================================
import os, glob, subprocess

# LD_LIBRARY_PATH phải trỏ tới thư viện CUDA/cuDNN mà torch cài qua pip,
# onnxruntime không tự tìm thấy chúng nên phải chỉ đường tường minh.
def nvidia_env():
    libs = sorted(set(glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib")))
    ld = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":")
    return dict(os.environ, LD_LIBRARY_PATH=ld), libs

# In output theo dòng ngay khi có, không đợi lệnh chạy xong
def run(cmd, env=None, cwd=None):
    p = subprocess.Popen(cmd, env=env or os.environ.copy(), cwd=cwd,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        print(f"\n[lệnh thất bại, exit code {p.returncode}]")
    return p.returncode == 0

ENV, LIBS = nvidia_env()
print("Thư viện NVIDIA tìm thấy:")
for p in LIBS:
    print("  ", p)
print()

import onnxruntime as ort
print("ONNX Runtime :", ort.__version__)
print("Providers    :", ort.get_available_providers())
print()
print("Chưa kiểm tra được CUDA thật lúc này vì chưa có file ONNX.")
print("Cell 8 sẽ in ra provider thực tế của từng session.")

In [ ]:
# ============================================================
# Cell 3: CLONE REPO INFERENCE
# ============================================================
import os

REPO = "/kaggle/working/voxcpm2_onnx"
if not os.path.exists(REPO):
    run(["git", "clone", "https://github.com/ai4all8/VoxCPM2-ONNX.git", REPO])
else:
    print("Repo đã có, bỏ qua clone.")

print("\nFiles trong repo:")
for f in sorted(os.listdir(REPO)):
    print("  ", f)

In [ ]:
# ============================================================
# Cell 4: TẢI ONNX MODELS VÀO ĐÚNG THƯ MỤC REPO
# ============================================================
# infer.py tìm model ở <repo>/onnx_models, tải ra chỗ khác sẽ báo thiếu file.
from huggingface_hub import snapshot_download
import os, glob, shutil

ONNX_DIR = os.path.join(REPO, "onnx_models")
NEEDED = ["voxcpm2_prefill.onnx", "voxcpm2_decode_step.onnx", "audio_vae_decoder.onnx"]

os.makedirs(ONNX_DIR, exist_ok=True)

if all(os.path.exists(os.path.join(ONNX_DIR, f)) for f in NEEDED):
    print("Đã có đủ file ONNX, bỏ qua tải.")
else:
    print("Đang tải VoxCPM2-ONNX (~17GB, khoảng 5-10 phút)...")
    snapshot_download("ai4all8/VoxCPM2-ONNX", local_dir=ONNX_DIR)

    # Nếu file nằm trong thư mục con thì kéo lên gốc
    for src in glob.glob(os.path.join(ONNX_DIR, "**", "*.onnx*"), recursive=True):
        dst = os.path.join(ONNX_DIR, os.path.basename(src))
        if src != dst and not os.path.exists(dst):
            shutil.move(src, dst)

print(f"\nFiles trong {ONNX_DIR}:")
for f in sorted(os.listdir(ONNX_DIR)):
    p = os.path.join(ONNX_DIR, f)
    if os.path.isfile(p):
        print(f"  {f}  ({os.path.getsize(p)/1e9:.2f} GB)")

missing = [f for f in NEEDED if not os.path.exists(os.path.join(ONNX_DIR, f))]
if missing:
    print(f"\nTHIẾU: {missing}")
    print(f"Thử chạy script của repo: !cd {REPO} && python download_models.py")
    raise SystemExit(1)
print("\nĐủ 3 file ONNX cần thiết.")

In [ ]:
# ============================================================
# Cell 5: TẢI VOXCPM2 BASE MODEL VÀO CACHE
# ============================================================
# Cần cho tokenizer và encode ref audio. Dùng snapshot_download chứ KHÔNG dùng
# VoxCPM.from_pretrained, vì lệnh đó nạp model lên GPU và giữ 5.5GB VRAM,
# khiến decode_step 8.66GB không còn chỗ.
from huggingface_hub import snapshot_download
import glob

cached = glob.glob("/root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/*")
if cached:
    print("Đã có trong cache:", cached[-1])
else:
    print("Đang tải VoxCPM2 base model...")
    snapshot_download("openbmb/VoxCPM2")
    print("Xong.")

print("VRAM đang dùng: 0 GB (chỉ tải file, không nạp model)")

In [ ]:
# ============================================================
# Cell 6: CẤU HÌNH REF AUDIO + CÂU TEST
# ============================================================
# ===== SỬA 2 DÒNG NÀY =====
REF_AUDIO = "/kaggle/input/my-voice/giong_cua_toi.wav"   # đường dẫn ref audio
REF_TEXT  = "Xin chào, đây là giọng nói của tôi."        # đúng nội dung đã đọc
# ==========================

TEXTS = [
    "Hôm nay thời tiết khá đẹp, rất thích hợp để đi dạo công viên.",
    "Cà phê sữa đá là thức uống yêu thích của rất nhiều người Việt Nam.",
    "Bộ Giáo dục và Đào tạo vừa công bố phương án thi tốt nghiệp.",
    "Chiều nay mình phải ghé siêu thị mua ít đồ dùng cho gia đình.",
    "Bạn đã ăn cơm chưa? Nếu chưa thì mình cùng đi ăn nhé.",
]

import json, librosa, os

assert os.path.exists(REF_AUDIO), f"Không tìm thấy ref audio: {REF_AUDIO}"
wave, sr = librosa.load(REF_AUDIO, sr=None, mono=True)
dur = len(wave) / sr

print(f"Ref audio : {REF_AUDIO}")
print(f"Thời lượng: {dur:.1f}s  |  Sample rate: {sr} Hz")
print(f"Transcript: {REF_TEXT}")
print(f"Số câu test: {len(TEXTS)}")

if not 3 <= dur <= 10:
    print(f"\nLưu ý: ref audio nên dài 3-10 giây, hiện tại {dur:.1f}s.")

json.dump({"ref_audio": REF_AUDIO, "ref_text": REF_TEXT, "texts": TEXTS},
          open("/kaggle/working/config.json", "w"), ensure_ascii=False)
print("\nĐã ghi config.json")

In [ ]:
%%writefile /kaggle/working/voxcpm2_onnx/prep.py
# Preprocessing bang PyTorch tren CPU, luu input cho ONNX ra npz.
# Chay rieng mot process voi CUDA_VISIBLE_DEVICES="" de khong chiem VRAM,
# va de model PyTorch duoc giai phong hoan toan truoc khi nap ONNX sessions.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import sys, json
import numpy as np
import torch

SCRIPT_DIR = "/kaggle/working/voxcpm2_onnx"
os.chdir(SCRIPT_DIR)
sys.path.insert(0, os.path.join(SCRIPT_DIR, "src"))
sys.path.insert(0, os.path.join(SCRIPT_DIR, "export"))

from infer import find_voxcpm2_model, build_prefill_inputs
from voxcpm.model.voxcpm2 import VoxCPM2Model

cfg = json.load(open("/kaggle/working/config.json"))

model_path = find_voxcpm2_model()
assert model_path, "Khong tim thay VoxCPM2 trong HF cache. Chay lai Cell 5."
print("Nap model tu:", model_path)

model = VoxCPM2Model.from_local(model_path, optimize=False)
# .cpu() TRUOC roi moi doi dtype. Lam nguoc lai se convert fp32 ngay tren GPU.
model = model.cpu().to(torch.float32).eval()

out = {}
for i, text in enumerate(cfg["texts"]):
    a, b, c, e = build_prefill_inputs(model, text, cfg["ref_audio"], cfg["ref_text"])
    out[f"text_{i}"], out[f"tm_{i}"], out[f"f_{i}"], out[f"fm_{i}"] = a, b, c, e
    print(f"  [{i}] xong")

np.savez("/kaggle/working/prepped.npz", n=np.array(len(cfg["texts"])), **out)
print("Da luu /kaggle/working/prepped.npz")

In [ ]:
# ============================================================
# Cell 7: CHẠY PREPROCESSING
# ============================================================
if os.path.exists("/kaggle/working/prepped.npz"):
    print("Đã có prepped.npz, bỏ qua. Xoá file đó nếu muốn chạy lại.")
else:
    ok = run(["python", "prep.py"],
             env=dict(os.environ, CUDA_VISIBLE_DEVICES=""),
             cwd=REPO)
    assert ok, "Preprocessing thất bại."

import numpy as np
d = np.load("/kaggle/working/prepped.npz")
print(f"\nSố câu đã chuẩn bị: {int(d['n'])}")

In [ ]:
%%writefile /kaggle/working/voxcpm2_onnx/bench.py
# Benchmark VoxCPM2 ONNX. Nap session MOT lan roi lap qua cac cau,
# khac voi infer.py goc moi cau mot process (RTF do duoc se la thoi gian nap model).
#   --mode gpu : decode_step + vae tren CUDA, prefill tren CPU
#   --mode cpu : tat ca tren CPU
import os, sys, time, json, argparse
import numpy as np
import soundfile as sf
import onnxruntime as ort

SCRIPT_DIR = "/kaggle/working/voxcpm2_onnx"
os.chdir(SCRIPT_DIR)
sys.path.insert(0, os.path.join(SCRIPT_DIR, "src"))
sys.path.insert(0, os.path.join(SCRIPT_DIR, "export"))
from inference.infer_loop import run_inference

OUT_SR = 48000
ONNX_DIR = os.path.join(SCRIPT_DIR, "onnx_models")

ap = argparse.ArgumentParser()
ap.add_argument("--mode", choices=["gpu", "cpu"], required=True)
args = ap.parse_args()

CPU = ["CPUExecutionProvider"]
CUDA = ["CUDAExecutionProvider", "CPUExecutionProvider"]

def mk(name, providers):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(os.path.join(ONNX_DIR, name), so, providers=providers)

# prefill 8.36GB + decode_step 8.66GB = 17GB, vuot 14.56GB VRAM cua T4.
# decode_step chay ~30 lan moi cau nen uu tien no len GPU; prefill chay 1 lan.
hot = CUDA if args.mode == "gpu" else CPU

print(f"[{args.mode}] Nap ONNX sessions (lan dau ~75s do graph optimization)...")
t0 = time.time()
prefill_sess = mk("voxcpm2_prefill.onnx", CPU)
decode_sess = mk("voxcpm2_decode_step.onnx", hot)
vae_sess = mk("audio_vae_decoder.onnx", hot)
load_time = time.time() - t0

providers = {
    "prefill": prefill_sess.get_providers()[0],
    "decode_step": decode_sess.get_providers()[0],
    "vae": vae_sess.get_providers()[0],
}
print(f"  Nap xong {load_time:.1f}s")
for k, v in providers.items():
    print(f"  {k:12}: {v}")

if args.mode == "gpu" and providers["decode_step"] != "CUDAExecutionProvider":
    print()
    print("  CANH BAO: decode_step da fallback ve CPU.")
    print("  Ket qua se giong het mode cpu. Kiem tra onnxruntime-gpu va LD_LIBRARY_PATH.")

d = np.load("/kaggle/working/prepped.npz")
n = int(d["n"])
vin = vae_sess.get_inputs()[0].name
vout = vae_sess.get_outputs()[0].name

print()
print(f"[{args.mode}] Benchmark {n} cau...")
results = []
for i in range(n):
    t0 = time.time()
    latents = run_inference(prefill_sess, decode_sess,
                            d[f"text_{i}"], d[f"tm_{i}"], d[f"f_{i}"], d[f"fm_{i}"],
                            min_len=5, max_len=300, cfg_value=2.0)
    audio = vae_sess.run([vout], {vin: latents})[0].squeeze()
    elapsed = time.time() - t0

    dur = len(audio) / OUT_SR
    rtf = elapsed / max(dur, 1e-6)
    sf.write(f"/kaggle/working/{args.mode}_bench_{i}.wav", audio, OUT_SR)
    results.append({"i": i, "gen_s": elapsed, "audio_s": dur, "rtf": rtf})
    tag = "   (warm-up, bo khi tinh trung binh)" if i == 0 else ""
    print(f"  [{i}] gen={elapsed:5.1f}s  audio={dur:4.1f}s  RTF={rtf:6.3f}{tag}")

steady = results[1:] if len(results) > 1 else results
summary = {
    "mode": args.mode,
    "providers": providers,
    "load_time": load_time,
    "results": results,
    "rtf_mean": float(np.mean([r["rtf"] for r in steady])),
    "rtf_min": float(min(r["rtf"] for r in steady)),
    "rtf_max": float(max(r["rtf"] for r in steady)),
}
json.dump(summary, open(f"/kaggle/working/bench_{args.mode}.json", "w"), indent=1)

print()
print(f"  RTF trung binh ({args.mode}): {summary['rtf_mean']:.4f}")

In [ ]:
# ============================================================
# Cell 8: BENCHMARK TRÊN GPU
# ============================================================
# Nhìn dòng "decode_step:" trong output. Nếu không phải CUDAExecutionProvider
# thì nó đã fallback về CPU và con số RTF sẽ không phản ánh GPU.
run(["python", "bench.py", "--mode", "gpu"], env=ENV, cwd=REPO)

In [ ]:
# ============================================================
# Cell 9: BENCHMARK TRÊN CPU (để so sánh)
# ============================================================
# Khoảng 7-8 phút. Bỏ qua được nếu chỉ cần số GPU.
run(["python", "bench.py", "--mode", "cpu"], env=ENV, cwd=REPO)

In [ ]:
# ============================================================
# Cell 10: BÁO CÁO
# ============================================================
import json, os

print("=" * 64)
print("BÁO CÁO: VoxCPM2 (2B) ONNX — RTF trên Kaggle T4")
print("=" * 64)

loaded = {}
for mode in ["gpu", "cpu"]:
    p = f"/kaggle/working/bench_{mode}.json"
    if os.path.exists(p):
        loaded[mode] = json.load(open(p))

for mode, s in loaded.items():
    print(f"\n--- mode = {mode} ---")
    print(f"  prefill      : {s['providers']['prefill']}")
    print(f"  decode_step  : {s['providers']['decode_step']}")
    print(f"  vae          : {s['providers']['vae']}")
    print(f"  Nạp sessions : {s['load_time']:.1f}s (một lần)")
    print(f"  RTF trung bình : {s['rtf_mean']:.4f}")
    print(f"  RTF thấp / cao : {s['rtf_min']:.4f} / {s['rtf_max']:.4f}")

if len(loaded) == 2:
    speedup = loaded["cpu"]["rtf_mean"] / loaded["gpu"]["rtf_mean"]
    print(f"\nGPU nhanh hơn CPU: {speedup:.1f} lần")

print("\n" + "-" * 64)
print("Ghi chú cho báo cáo: mỗi câu cần khoảng 30 bước autoregressive, mỗi bước")
print("chứa 10 timesteps diffusion, tổng khoảng 300 lần forward qua model 2B.")
print("Chuỗi này tuần tự nên ONNX không song song hoá được. Mốc RTF 0.05 không")
print("đạt được bằng cách tối ưu runtime, phải đổi kiến trúc model.")

In [ ]:
# ============================================================
# Cell 11: NGHE THỬ
# ============================================================
from IPython.display import Audio, display
import glob

for mode in ["gpu", "cpu"]:
    files = sorted(glob.glob(f"/kaggle/working/{mode}_bench_*.wav"))
    if not files:
        continue
    # bỏ file 0 vì đó là câu warm-up
    pick = files[1] if len(files) > 1 else files[0]
    print(f"--- {mode}: {os.path.basename(pick)} ---")
    display(Audio(pick))